# Phase 1: Data Processing Pipeline for Twi/Akan Voice Technology

This notebook implements comprehensive data processing for voice datasets:
1. Audio processing (16kHz conversion, normalization, VAD)
2. Text processing (Twi character handling, normalization)
3. Data validation and quality checks
4. Train/validation/test splitting
5. Data augmentation techniques

## 1. Setup and Dependencies

In [ ]:
# Install required packages
!pip install -q librosa soundfile webrtcvad pydub
!pip install -q pandas numpy tqdm
!pip install -q num2words unidecode
!pip install -q audiomentations

In [ ]:
import os
import sys
import json
import shutil
import librosa
import soundfile as sf
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from tqdm.auto import tqdm
import webrtcvad
from pydub import AudioSegment
import warnings
import unicodedata
import re
from collections import Counter
warnings.filterwarnings('ignore')

# Check environment
IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'kaggle_secrets' in sys.modules

print(f"Environment: {'Google Colab' if IN_COLAB else 'Kaggle' if IN_KAGGLE else 'Local'}")
print(f"Python: {sys.version}")

In [ ]:
# Setup paths
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/voice-lab-data')
elif IN_KAGGLE:
    BASE_DIR = Path('/kaggle/working/voice-lab-data')
else:
    BASE_DIR = Path('./data')

# Create directory structure
DIRS = {
    'raw': BASE_DIR / 'raw',
    'processed': BASE_DIR / 'processed',
    'splits': BASE_DIR / 'splits',
    'augmented': BASE_DIR / 'augmented',
    'metadata': BASE_DIR / 'metadata'
}

for dir_path in DIRS.values():
    dir_path.mkdir(parents=True, exist_ok=True)

print(f"Base directory: {BASE_DIR}")
print("Directory structure created.")

## 2. Audio Processing Pipeline

In [ ]:
class AudioProcessor:
    """Comprehensive audio processing for ASR/TTS training."""
    
    def __init__(self, target_sr=16000, target_db=-3.0):
        self.target_sr = target_sr
        self.target_db = target_db
        self.vad = webrtcvad.Vad(2)  # Aggressiveness level 2
        
    def process_audio_file(self, input_path: str, output_path: str) -> Dict:
        """Process single audio file with all transformations."""
        try:
            # Load audio
            audio, sr = librosa.load(input_path, sr=None)
            original_duration = len(audio) / sr
            
            # Resample if needed
            if sr != self.target_sr:
                audio = librosa.resample(audio, orig_sr=sr, target_sr=self.target_sr)
                sr = self.target_sr
            
            # Remove silence
            audio_trimmed = self.remove_silence(audio, sr)
            
            # Normalize audio
            audio_normalized = self.normalize_audio(audio_trimmed)
            
            # Apply VAD
            audio_vad = self.apply_vad(audio_normalized, sr)
            
            # Save processed audio
            sf.write(output_path, audio_vad, sr)
            
            # Return metadata
            return {
                'original_duration': original_duration,
                'processed_duration': len(audio_vad) / sr,
                'sample_rate': sr,
                'removed_silence': original_duration - len(audio_vad) / sr,
                'peak_amplitude': np.max(np.abs(audio_vad))
            }
            
        except Exception as e:
            print(f"Error processing {input_path}: {e}")
            return None
    
    def remove_silence(self, audio: np.ndarray, sr: int, 
                      top_db: int = 30) -> np.ndarray:
        """Remove silence from beginning and end."""
        intervals = librosa.effects.split(audio, top_db=top_db)
        if len(intervals) > 0:
            start = intervals[0][0]
            end = intervals[-1][1]
            return audio[start:end]
        return audio
    
    def normalize_audio(self, audio: np.ndarray) -> np.ndarray:
        """Normalize audio to target dB."""
        # Calculate current dB
        rms = np.sqrt(np.mean(audio**2))
        current_db = 20 * np.log10(rms) if rms > 0 else -np.inf
        
        # Calculate scaling factor
        target_rms = 10**(self.target_db / 20)
        scaling_factor = target_rms / rms if rms > 0 else 1.0
        
        # Apply scaling
        audio_normalized = audio * scaling_factor
        
        # Prevent clipping
        max_val = np.max(np.abs(audio_normalized))
        if max_val > 0.95:
            audio_normalized = audio_normalized * (0.95 / max_val)
        
        return audio_normalized
    
    def apply_vad(self, audio: np.ndarray, sr: int) -> np.ndarray:
        """Apply Voice Activity Detection."""
        # Convert to 16-bit PCM
        audio_int16 = (audio * 32767).astype(np.int16)
        
        # Frame size for VAD (10, 20, or 30 ms)
        frame_duration_ms = 30
        frame_length = int(sr * frame_duration_ms / 1000)
        
        # Apply VAD frame by frame
        voiced_frames = []
        for i in range(0, len(audio_int16), frame_length):
            frame = audio_int16[i:i+frame_length]
            if len(frame) < frame_length:
                frame = np.pad(frame, (0, frame_length - len(frame)))
            
            if self.vad.is_speech(frame.tobytes(), sr):
                voiced_frames.append(audio[i:i+frame_length])
        
        if voiced_frames:
            return np.concatenate(voiced_frames)
        return audio
    
    def segment_audio(self, audio: np.ndarray, sr: int, 
                     max_duration: float = 30.0) -> List[np.ndarray]:
        """Segment long audio into chunks."""
        max_samples = int(max_duration * sr)
        segments = []
        
        for i in range(0, len(audio), max_samples):
            segment = audio[i:i+max_samples]
            if len(segment) > sr:  # At least 1 second
                segments.append(segment)
        
        return segments

## 3. Text Processing Pipeline

In [ ]:
class TwiTextProcessor:
    """Text processing specifically for Twi/Akan languages."""
    
    def __init__(self):
        # Twi-specific characters
        self.twi_chars = {
            'ɛ', 'Ɛ', 'ɔ', 'Ɔ', 'ã', 'ẽ', 'ĩ', 'õ', 'ũ'
        }
        
        # Character normalization mapping
        self.char_map = {
            'è': 'ɛ', 'È': 'Ɛ',
            'ò': 'ɔ', 'Ò': 'Ɔ',
            'e': 'e', 'E': 'E',  # Sometimes confused
            'o': 'o', 'O': 'O',  # Sometimes confused
        }
        
        # Common code-switching patterns (Twi-English)
        self.english_words = set([
            'okay', 'yes', 'no', 'hello', 'please', 'thank', 'you',
            'sorry', 'excuse', 'me', 'good', 'morning', 'afternoon',
            'evening', 'night', 'bye', 'goodbye'
        ])
        
    def process_text(self, text: str) -> Dict:
        """Process Twi text with all transformations."""
        original = text
        
        # Normalize unicode
        text = unicodedata.normalize('NFC', text)
        
        # Handle special Twi characters
        text = self.normalize_twi_chars(text)
        
        # Normalize punctuation
        text = self.normalize_punctuation(text)
        
        # Handle code-switching
        text, code_switches = self.detect_code_switching(text)
        
        # Standardize spelling
        text = self.standardize_spelling(text)
        
        # Remove extra spaces
        text = ' '.join(text.split())
        
        return {
            'original': original,
            'processed': text,
            'has_twi_chars': any(c in text for c in self.twi_chars),
            'code_switches': code_switches,
            'char_count': len(text),
            'word_count': len(text.split())
        }
    
    def normalize_twi_chars(self, text: str) -> str:
        """Normalize Twi-specific characters."""
        for old_char, new_char in self.char_map.items():
            text = text.replace(old_char, new_char)
        return text
    
    def normalize_punctuation(self, text: str) -> str:
        """Normalize punctuation marks."""
        # Replace smart quotes using character codes
        text = text.replace(chr(8220), chr(34)).replace(chr(8221), chr(34))
        text = text.replace(chr(8216), chr(39)).replace(chr(8217), chr(39))
        
        # Remove extra spaces around punctuation
        text = re.sub(r'\s+([,.!?;:])', r'\1', text)
        text = re.sub(r'([,.!?;:])\s+', r'\1 ', text)
        
        return text
    
    def detect_code_switching(self, text: str) -> Tuple[str, List[str]]:
        """Detect and handle code-switching."""
        words = text.lower().split()
        code_switches = []
        
        for word in words:
            if word in self.english_words:
                code_switches.append(word)
        
        return text, code_switches
    
    def standardize_spelling(self, text: str) -> str:
        """Standardize common spelling variations."""
        # Common Twi spelling variations
        replacements = {
            'wo ho': 'woho',  # greeting
            'me da ase': 'medaase',  # thank you
            'ɛyɛ': 'ɛyɛ',  # it is
        }
        
        for old, new in replacements.items():
            text = text.replace(old, new)
        
        return text
    
    def create_phoneme_mapping(self, text: str) -> str:
        """Create basic phoneme mapping for Twi."""
        # Simplified phoneme mapping
        phoneme_map = {
            'ɛ': 'EH', 'Ɛ': 'EH',
            'ɔ': 'OH', 'Ɔ': 'OH',
            'ã': 'AN', 'ẽ': 'EN',
            'ĩ': 'IN', 'õ': 'ON',
            'ũ': 'UN'
        }
        
        phonemes = text
        for char, phoneme in phoneme_map.items():
            phonemes = phonemes.replace(char, f'[{phoneme}]')
        
        return phonemes

## 4. Data Validation

In [ ]:
class DataValidator:
    """Validate audio-text pairs for quality."""
    
    def __init__(self, min_duration=0.5, max_duration=30.0, min_words=2):
        self.min_duration = min_duration
        self.max_duration = max_duration
        self.min_words = min_words
    
    def validate_pair(self, audio_path: str, text: str) -> Dict:
        """Validate single audio-text pair."""
        issues = []
        
        # Check audio file
        if not Path(audio_path).exists():
            issues.append('audio_file_missing')
            return {'valid': False, 'issues': issues}
        
        try:
            # Load audio and check duration
            audio, sr = librosa.load(audio_path, sr=None)
            duration = len(audio) / sr
            
            if duration < self.min_duration:
                issues.append('audio_too_short')
            if duration > self.max_duration:
                issues.append('audio_too_long')
            
            # Check for silence
            if np.max(np.abs(audio)) < 0.01:
                issues.append('audio_silent')
            
        except Exception as e:
            issues.append(f'audio_error: {e}')
        
        # Check text
        if not text or len(text.strip()) == 0:
            issues.append('text_empty')
        elif len(text.split()) < self.min_words:
            issues.append('text_too_short')
        
        # Check for alignment (words per second)
        if not issues and duration > 0:
            words_per_second = len(text.split()) / duration
            if words_per_second > 10:  # Too fast
                issues.append('possible_misalignment_fast')
            elif words_per_second < 0.5:  # Too slow
                issues.append('possible_misalignment_slow')
        
        return {
            'valid': len(issues) == 0,
            'issues': issues,
            'duration': duration if 'duration' in locals() else None,
            'text_length': len(text),
            'word_count': len(text.split())
        }

## 5. Dataset Splitting

In [ ]:
def create_splits(data_df: pd.DataFrame, 
                 train_ratio: float = 0.7, 
                 val_ratio: float = 0.15,
                 test_ratio: float = 0.15,
                 stratify_by: str = None,
                 seed: int = 42) -> Dict[str, pd.DataFrame]:
    """Create train/val/test splits with optional stratification."""
    
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 0.001
    
    np.random.seed(seed)
    
    # Shuffle data
    data_df = data_df.sample(frac=1, random_state=seed).reset_index(drop=True)
    
    if stratify_by and stratify_by in data_df.columns:
        # Stratified split
        splits = {'train': [], 'validation': [], 'test': []}
        
        for group in data_df[stratify_by].unique():
            group_data = data_df[data_df[stratify_by] == group]
            n = len(group_data)
            
            train_end = int(n * train_ratio)
            val_end = int(n * (train_ratio + val_ratio))
            
            splits['train'].append(group_data.iloc[:train_end])
            splits['validation'].append(group_data.iloc[train_end:val_end])
            splits['test'].append(group_data.iloc[val_end:])
        
        return {
            'train': pd.concat(splits['train'], ignore_index=True),
            'validation': pd.concat(splits['validation'], ignore_index=True),
            'test': pd.concat(splits['test'], ignore_index=True)
        }
    else:
        # Simple split
        n = len(data_df)
        train_end = int(n * train_ratio)
        val_end = int(n * (train_ratio + val_ratio))
        
        return {
            'train': data_df.iloc[:train_end],
            'validation': data_df.iloc[train_end:val_end],
            'test': data_df.iloc[val_end:]
        }

def save_splits(splits: Dict[str, pd.DataFrame], output_dir: Path):
    """Save splits to CSV files."""
    for split_name, split_df in splits.items():
        output_path = output_dir / f'{split_name}.csv'
        split_df.to_csv(output_path, index=False)
        print(f"Saved {split_name}: {len(split_df)} samples to {output_path}")

## 6. Data Augmentation

In [ ]:
class AudioAugmenter:
    """Audio augmentation for training data."""
    
    def __init__(self):
        try:
            import audiomentations as A
            self.augmentations = A.Compose([
                A.TimeStretch(min_rate=0.9, max_rate=1.1, p=0.3),
                A.PitchShift(min_semitones=-2, max_semitones=2, p=0.3),
                A.AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.015, p=0.2),
                A.AddGaussianSNR(min_snr_in_db=10, max_snr_in_db=20, p=0.2),
            ])
            self.has_audiomentations = True
        except ImportError:
            self.has_audiomentations = False
            print("audiomentations not available, using basic augmentation")
    
    def augment_audio(self, audio: np.ndarray, sr: int, 
                     augmentation_type: str = 'random') -> np.ndarray:
        """Apply augmentation to audio."""
        
        if self.has_audiomentations and augmentation_type == 'random':
            return self.augmentations(samples=audio, sample_rate=sr)
        
        # Basic augmentations without audiomentations
        if augmentation_type == 'speed':
            return self.speed_perturbation(audio, sr)
        elif augmentation_type == 'pitch':
            return self.pitch_shift(audio, sr)
        elif augmentation_type == 'noise':
            return self.add_noise(audio)
        else:
            return audio
    
    def speed_perturbation(self, audio: np.ndarray, sr: int, 
                          rate: float = None) -> np.ndarray:
        """Change playback speed."""
        if rate is None:
            rate = np.random.uniform(0.9, 1.1)
        return librosa.effects.time_stretch(audio, rate=rate)
    
    def pitch_shift(self, audio: np.ndarray, sr: int, 
                   steps: float = None) -> np.ndarray:
        """Shift pitch."""
        if steps is None:
            steps = np.random.uniform(-2, 2)
        return librosa.effects.pitch_shift(audio, sr=sr, n_steps=steps)
    
    def add_noise(self, audio: np.ndarray, noise_level: float = 0.01) -> np.ndarray:
        """Add Gaussian noise."""
        noise = np.random.normal(0, noise_level, audio.shape)
        return audio + noise
    
    def add_room_reverb(self, audio: np.ndarray, sr: int) -> np.ndarray:
        """Simulate room acoustics (simplified)."""
        # Simple reverb using delay and decay
        delay_samples = int(0.05 * sr)  # 50ms delay
        decay = 0.3
        
        reverb = np.zeros_like(audio)
        reverb[delay_samples:] = audio[:-delay_samples] * decay
        
        return audio + reverb

## 7. Main Processing Pipeline

In [ ]:
def process_dataset(dataset_name: str, dataset_path: Path) -> pd.DataFrame:
    """Process a complete dataset."""
    
    print(f"\nProcessing {dataset_name}...")
    
    # Initialize processors
    audio_processor = AudioProcessor()
    text_processor = TwiTextProcessor()
    validator = DataValidator()
    
    # Find all audio files
    audio_extensions = ['.wav', '.mp3', '.flac', '.ogg', '.m4a']
    audio_files = []
    for ext in audio_extensions:
        audio_files.extend(list(dataset_path.rglob(f'*{ext}')))
    
    print(f"Found {len(audio_files)} audio files")
    
    # Process each file
    results = []
    
    for audio_file in tqdm(audio_files, desc="Processing audio"):
        # Look for corresponding text file
        text_file = audio_file.with_suffix('.txt')
        
        if not text_file.exists():
            # Try to find text in metadata
            continue
        
        # Read text
        with open(text_file, 'r', encoding='utf-8') as f:
            text = f.read().strip()
        
        # Validate pair
        validation = validator.validate_pair(str(audio_file), text)
        
        if not validation['valid']:
            print(f"Skipping {audio_file.name}: {validation['issues']}")
            continue
        
        # Process audio
        output_audio = DIRS['processed'] / dataset_name / audio_file.name
        output_audio.parent.mkdir(parents=True, exist_ok=True)
        
        audio_meta = audio_processor.process_audio_file(
            str(audio_file), 
            str(output_audio)
        )
        
        # Process text
        text_meta = text_processor.process_text(text)
        
        # Store result
        results.append({
            'dataset': dataset_name,
            'audio': str(output_audio),
            'text': text_meta['processed'],
            'duration': audio_meta['processed_duration'] if audio_meta else None,
            'has_twi_chars': text_meta['has_twi_chars'],
            'word_count': text_meta['word_count'],
            'original_audio': str(audio_file)
        })
    
    return pd.DataFrame(results)

## 8. Process All Downloaded Datasets

In [ ]:
# Process each dataset
all_data = []

# Process BibleTTS datasets
bibletts_datasets = ['asante_twi', 'akuapem_twi']
for dialect in bibletts_datasets:
    dataset_path = DIRS['raw'] / 'bibletts' / dialect
    if dataset_path.exists():
        df = process_dataset(f'bibletts_{dialect}', dataset_path)
        all_data.append(df)

# Process Financial Inclusion dataset
financial_path = DIRS['raw'] / 'financial_inclusion'
if financial_path.exists():
    df = process_dataset('financial_inclusion', financial_path)
    all_data.append(df)

# Combine all processed data
if all_data:
    combined_df = pd.concat(all_data, ignore_index=True)
    print(f"\nTotal processed samples: {len(combined_df)}")
    
    # Save combined metadata
    combined_df.to_csv(DIRS['metadata'] / 'processed_data.csv', index=False)
else:
    print("No data to process. Please run the data acquisition notebook first.")

## 9. Create Train/Val/Test Splits

In [ ]:
if 'combined_df' in locals() and len(combined_df) > 0:
    # Create splits
    splits = create_splits(
        combined_df,
        train_ratio=0.7,
        val_ratio=0.15,
        test_ratio=0.15,
        stratify_by='dataset',  # Stratify by dataset to ensure balance
        seed=42
    )
    
    # Save splits
    save_splits(splits, DIRS['splits'])
    
    # Print statistics
    print("\nSplit Statistics:")
    for split_name, split_df in splits.items():
        print(f"\n{split_name.upper()}:")
        print(f"  Total samples: {len(split_df)}")
        print(f"  Total duration: {split_df['duration'].sum():.2f} seconds")
        print(f"  Datasets: {split_df['dataset'].value_counts().to_dict()}")
        print(f"  Has Twi chars: {split_df['has_twi_chars'].sum()} samples")

## 10. Generate Augmented Data

In [ ]:
# Generate augmented versions of training data
if 'splits' in locals():
    augmenter = AudioAugmenter()
    train_df = splits['train']
    
    # Select subset for augmentation (e.g., 20% of training data)
    augment_samples = train_df.sample(frac=0.2, random_state=42)
    
    print(f"Augmenting {len(augment_samples)} samples...")
    
    augmented_data = []
    for _, row in tqdm(augment_samples.iterrows(), total=len(augment_samples)):
        audio_path = row['audio']
        
        if Path(audio_path).exists():
            # Load audio
            audio, sr = librosa.load(audio_path, sr=16000)
            
            # Apply different augmentations
            for aug_type in ['speed', 'pitch', 'noise']:
                aug_audio = augmenter.augment_audio(audio, sr, aug_type)
                
                # Save augmented audio
                aug_path = DIRS['augmented'] / f"{Path(audio_path).stem}_{aug_type}.wav"
                aug_path.parent.mkdir(parents=True, exist_ok=True)
                sf.write(aug_path, aug_audio, sr)
                
                # Add to augmented data
                aug_row = row.copy()
                aug_row['audio'] = str(aug_path)
                aug_row['augmentation'] = aug_type
                augmented_data.append(aug_row)
    
    # Create augmented dataframe
    if augmented_data:
        augmented_df = pd.DataFrame(augmented_data)
        augmented_df.to_csv(DIRS['metadata'] / 'augmented_data.csv', index=False)
        print(f"Created {len(augmented_df)} augmented samples")

## Summary

In [ ]:
print("\n" + "="*60)
print("DATA PROCESSING COMPLETE")
print("="*60)

if 'combined_df' in locals():
    print(f"\nProcessed {len(combined_df)} total samples")
    print(f"Total duration: {combined_df['duration'].sum()/3600:.2f} hours")
    print(f"\nDatasets processed:")
    for dataset in combined_df['dataset'].unique():
        dataset_df = combined_df[combined_df['dataset'] == dataset]
        print(f"  - {dataset}: {len(dataset_df)} samples")
    
    print(f"\nText statistics:")
    print(f"  Samples with Twi characters: {combined_df['has_twi_chars'].sum()}")
    print(f"  Average word count: {combined_df['word_count'].mean():.1f}")
    
    if 'augmented_df' in locals():
        print(f"\nAugmentation:")
        print(f"  Generated {len(augmented_df)} augmented samples")
    
    print(f"\nOutput locations:")
    print(f"  Processed audio: {DIRS['processed']}")
    print(f"  Splits: {DIRS['splits']}")
    print(f"  Metadata: {DIRS['metadata']}")
    print(f"  Augmented: {DIRS['augmented']}")
    
    print("\nReady for Phase 2: ASR Development!")
else:
    print("\nNo data processed. Please run data acquisition first.")